# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata (Croissant schema)
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Keywords:", metadata.keywords)
print("Identifier:", metadata.identifier)
print("Version:", metadata.version)
print("License:", metadata.license)


## 2. Data Overview
Review available record sets and their fields, referencing by their `@id` values.

The FAIR² dataset contains multiple survey record sets. We'll enumerate the available record sets and fields, using the Croissant schema's `@id` for each entity.

In [ ]:
# List all record sets in the dataset using their @id
record_sets = metadata.recordSet

all_record_set_ids = []
for rs in record_sets:
    print(f"RecordSet Name: {getattr(rs, 'name', 'unknown')}  |  @id: {rs['@id']}")
    all_record_set_ids.append(rs['@id'])

# For each record set, display its fields and columns referenced by their @id
for rs in record_sets:
    print(f"\nFields in RecordSet (@id: {rs['@id']}, name: {getattr(rs, 'name', 'unknown')}):")
    fields = getattr(rs, 'field', [])
    for field in fields:
        field_id = field['@id']
        field_name = getattr(field, 'name', 'unknown')
        field_type = getattr(field, 'dataType', 'unknown')
        print(f"  - Field Name: {field_name}  |  @id: {field_id}  |  Type: {field_type}")
        columns = getattr(field, 'column', [])
        if columns:
            print(f"    Columns:")
            for col in columns:
                col_id = col['@id']
                col_name = getattr(col, 'name', 'unknown')
                print(f"      -- Column Name: {col_name}  |  @id: {col_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the `@id` fields from the previous overview to access records.

In [ ]:
# List available record sets' @id
pprint.pprint(all_record_set_ids)

# Choose the main record set @id for analysis (example: using the first record set)
main_record_set_id = all_record_set_ids[0]
print(f"\nExtracting records from: {main_record_set_id}")

# Load records from each record set into a DataFrame
dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns (field @ids) from main record set
print("Fields (as DataFrame columns):", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Here, we process the main record set.

- We'll filter survey participants by a numeric field (e.g. GAD-7 score).
- Normalize scores and group by a categorical demographic field.

All fields are referenced by their Croissant `@id`.

In [ ]:
# Example: Choose numeric and group fields by their @id
# Replace these with actual @id values for your dataset fields

# For demonstration, let's assume:
# - GAD-7 score field @id: 'https://api.app.sen.science/frontiers/7160411/gad7_score'
# - Group by 'village' (@id): 'https://api.app.sen.science/frontiers/7160411/village'

# If these field IDs don't exist, replace with those enumerated previously.
numeric_field_id = 'https://api.app.sen.science/frontiers/7160411/gad7_score'
group_field_id = 'https://api.app.sen.science/frontiers/7160411/village'

# Filtering records on GAD-7 score > 10
threshold = 10
df = dataframes[main_record_set_id]
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score normalization)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by village and show mean scores
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in main record set columns:", df.columns.tolist())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll display a histogram of GAD-7 scores, and show the mean score by village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 scores
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

    # Barplot: mean GAD-7 score by village
    if group_field_id in df.columns:
        village_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(12,4))
        sns.barplot(data=village_means, x=group_field_id, y=numeric_field_id, color='lightgreen')
        plt.title('Mean GAD-7 Score by Village')
        plt.xlabel('Village')
        plt.ylabel('Mean GAD-7 Score')
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"Cannot visualize. Field {numeric_field_id} not found.")

## 6. Conclusion
In this notebook, we:
- Loaded dataset metadata and records from a Croissant schema using `mlcroissant`.
- Enumerated record sets and their fields by `@id`.
- Extracted survey data into DataFrames, referencing all entities by Croissant `@id`.
- Performed filtering, normalization, and grouping for exploratory analysis.
- Visualized the distribution of key mental health indicators (GAD-7 scores).

This process demonstrates how to use AI-ready Croissant datasets with standardized metadata for reproducible, scalable research.